# 02 Demo CarDD YOLO11 Segmentation

This notebook loads the trained model from Drive and runs prediction demos.

Dependency order:

1. Run **Mount Drive**.
2. Run **Install and Import**.
3. Run **Configure Paths**.
4. Run **Predict** after `best.pt` exists from notebook 01.

## Mount Drive

Dependency: none.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install and Import

Dependency: Drive is mounted.

In [ ]:
%pip install -q ultralytics opencv-python matplotlib pandas pyyaml

In [ ]:
from pathlib import Path
import random

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

## Configure Paths

Dependency: imports are ready. Put your own demo images into `demo_images`; otherwise validation/test samples are used.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CarDD_YOLO11')
RUN_NAME = 'yolo11n_seg'
DATA_YOLO_ROOT = DRIVE_ROOT / 'data_yolo'
TRAIN_DIR = DRIVE_ROOT / 'runs' / 'train' / RUN_NAME
PREDICT_PROJECT = DRIVE_ROOT / 'runs' / 'predict'
DEMO_IMAGE_DIR = DRIVE_ROOT / 'demo_images'

WEIGHTS = TRAIN_DIR / 'weights' / 'best.pt'
if not WEIGHTS.exists():
    WEIGHTS = TRAIN_DIR / 'weights' / 'last.pt'

DEMO_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PREDICT_PROJECT.mkdir(parents=True, exist_ok=True)

print('Weights:', WEIGHTS)
print('Put optional demo images here:', DEMO_IMAGE_DIR)

## Select Demo Images

Dependency: trained weights exist. If `demo_images` is empty, this cell samples from the prepared YOLO val/test images.

In [ ]:
assert WEIGHTS.exists(), 'No best.pt or last.pt found. Run notebook 01 first.'

exts = {'.jpg', '.jpeg', '.png', '.bmp'}
demo_images = [p for p in DEMO_IMAGE_DIR.glob('*') if p.suffix.lower() in exts]

if not demo_images:
    candidates = []
    for split in ['test', 'val', 'train']:
        split_dir = DATA_YOLO_ROOT / 'images' / split
        if split_dir.exists():
            candidates.extend([p for p in split_dir.glob('*') if p.suffix.lower() in exts])
        if candidates:
            break
    demo_images = random.sample(candidates, k=min(8, len(candidates)))

assert demo_images, 'No demo images found. Put images in Drive demo_images or prepare the dataset first.'
print('Demo images:')
for path in demo_images:
    print(path)

## Predict

Dependency: demo image list exists. Outputs are saved to Drive under `runs/predict/demo`.

In [ ]:
model = YOLO(str(WEIGHTS))
results = model.predict(
    source=[str(p) for p in demo_images],
    imgsz=1024,
    conf=0.25,
    iou=0.7,
    retina_masks=True,
    save=True,
    save_txt=True,
    save_conf=True,
    project=str(PREDICT_PROJECT),
    name='demo',
    exist_ok=True,
)

print('Prediction output:', PREDICT_PROJECT / 'demo')

## Show Saved Predictions

Dependency: Predict has run. This displays the saved visualization images from Drive.

In [ ]:
pred_dir = PREDICT_PROJECT / 'demo'
pred_images = [p for p in pred_dir.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
for path in pred_images[:8]:
    image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.title(path.name)
    plt.show()